In [1]:
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
data = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")

In [3]:
data = data.dropna()

In [4]:
X = data.drop("target", axis=1).to_numpy(dtype=float)
y = data["target"].to_numpy(dtype=int)

In [5]:
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

In [6]:
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

feature_names = data.drop("target", axis=1).columns

In [7]:
def gini(y):
    if len(y) == 0:
        return 0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p ** 2)

In [8]:
def build_rf_tree(X, y, depth=0, importance=None):
    classes, counts = np.unique(y, return_counts=True)
    prediction = classes[np.argmax(counts)]

    if len(classes) == 1 or depth >= 5 or len(y) < 4:
        return prediction

    best_gain = 0
    best_feature = None
    best_value = None

    features = np.random.choice(
        X.shape[1],
        max(1, int(np.sqrt(X.shape[1]))),
        replace=False
    )

    for feature in features:
        values = np.unique(X[:, feature])

        for value in values[:-1]:
            left = X[:, feature] <= value
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            gain = gini(y) - (
                left.mean() * gini(y[left])
                + right.mean() * gini(y[right])
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_value = value

    if best_feature is None:
        return prediction

    left = X[:, best_feature] <= best_value
    right = ~left

    importance[best_feature] += best_gain * len(y)

    return {
        "feature": best_feature,
        "value": best_value,
        "prediction": prediction,
        "left": build_rf_tree(
            X[left], y[left], depth + 1, importance
        ),
        "right": build_rf_tree(
            X[right], y[right], depth + 1, importance
        )
    }

In [9]:
def predict_tree(tree, row):
    if not isinstance(tree, dict):
        return tree

    if row[tree["feature"]] <= tree["value"]:
        return predict_tree(tree["left"], row)

    return predict_tree(tree["right"], row)

In [10]:
np.random.seed(42)
rf_trees = []
rf_importance = np.zeros(X.shape[1])

for i in range(10):
    sample = np.random.choice(
        len(X_train), len(X_train), replace=True
    )

    tree = build_rf_tree(
        X_train[sample], y_train[sample],
        importance=rf_importance
    )
    rf_trees.append(tree)

In [11]:
rf_predictions = []

for row in X_test:
    votes = [predict_tree(tree, row) for tree in rf_trees]
    rf_predictions.append(Counter(votes).most_common(1)[0][0])

In [12]:


def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1 / (1 + np.exp(-x))

In [13]:
def build_boost_tree(X, grad, hess, depth=0,
                     max_depth=2, reg=1.0, importance=None):

    G = grad.sum()
    H = hess.sum()

    leaf_value = -G / (H + reg)

    if depth >= max_depth or len(grad) < 5:
        return leaf_value

    best_gain = 0
    best_feature = None
    best_value = None

    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])

        for value in values[:-1]:
            left = X[:, feature] <= value
            right = ~left

            if left.sum() < 2 or right.sum() < 2:
                continue

            GL, HL = grad[left].sum(), hess[left].sum()
            GR, HR = grad[right].sum(), hess[right].sum()

            gain = 0.5 * (
                GL ** 2 / (HL + reg)
                + GR ** 2 / (HR + reg)
                - G ** 2 / (H + reg)
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_value = value

    if best_feature is None:
        return leaf_value

    left = X[:, best_feature] <= best_value
    right = ~left

    importance[best_feature] += best_gain

    return {
        "feature": best_feature,
        "value": best_value,
        "left": build_boost_tree(
            X[left], grad[left], hess[left],
            depth + 1, max_depth, reg, importance
        ),
        "right": build_boost_tree(
            X[right], grad[right], hess[right],
            depth + 1, max_depth, reg, importance
        )
    }

In [14]:
def predict_boost_tree(tree, row):
    if not isinstance(tree, dict):
        return tree

    if row[tree["feature"]] <= tree["value"]:
        return predict_boost_tree(tree["left"], row)

    return predict_boost_tree(tree["right"], row)

In [15]:
learning_rate = 0.1
number_of_trees = 30

positive_rate = np.clip(y_train.mean(), 1e-6, 1 - 1e-6)
base_score = np.log(positive_rate / (1 - positive_rate))

train_scores = np.full(len(X_train), base_score)
test_scores = np.full(len(X_test), base_score)

boost_trees = []
xgb_importance = np.zeros(X.shape[1])

for i in range(number_of_trees):
    probabilities = sigmoid(train_scores)

    gradients = probabilities - y_train
    hessians = probabilities * (1 - probabilities)

    tree = build_boost_tree(
        X_train, gradients, hessians,
        max_depth=2, reg=1.0,
        importance=xgb_importance
    )

    boost_trees.append(tree)

    train_scores += learning_rate * np.array([
        predict_boost_tree(tree, row) for row in X_train
    ])

    test_scores += learning_rate * np.array([
        predict_boost_tree(tree, row) for row in X_test
    ])

In [16]:
xgb_probabilities = sigmoid(test_scores)
xgb_predictions = (xgb_probabilities >= 0.5).astype(int)

In [17]:
def accuracy(actual, predicted):
    correct = 0

    for a, p in zip(actual, predicted):
        if a == p:
            correct += 1

    return correct / len(actual) * 100


rf_accuracy = accuracy(y_test, rf_predictions)
xgb_accuracy = accuracy(y_test, xgb_predictions)

print("Random Forest Accuracy:", round(rf_accuracy, 2), "%")
print("XGBoost-style Accuracy:", round(xgb_accuracy, 2), "%")

Random Forest Accuracy: 84.88 %
XGBoost-style Accuracy: 83.9 %


In [18]:
rf_importance = rf_importance / (rf_importance.sum() + 1e-12)
xgb_importance = xgb_importance / (xgb_importance.sum() + 1e-12)

importance_table = pd.DataFrame({
    "Feature": feature_names,
    "Random Forest": rf_importance,
    "XGBoost": xgb_importance
})

importance_table = importance_table.sort_values(
    "XGBoost", ascending=False
)

print("\nFeature Importance Comparison:")
print(importance_table.to_string(index=False))

print("\nTop Random Forest Features:")
print(
    importance_table.sort_values(
        "Random Forest", ascending=False
    )[["Feature", "Random Forest"]].head(5).to_string(index=False)
)

print("\nTop XGBoost Features:")
print(
    importance_table[["Feature", "XGBoost"]]
    .head(5).to_string(index=False)
)


Feature Importance Comparison:
 Feature  Random Forest  XGBoost
      cp       0.197879 0.279240
    thal       0.049351 0.212163
      ca       0.198878 0.190668
 oldpeak       0.101776 0.077766
   exang       0.084933 0.058787
     age       0.067221 0.043884
 thalach       0.127055 0.041471
     sex       0.034308 0.037348
   slope       0.041697 0.029354
    chol       0.044296 0.018020
 restecg       0.012431 0.006727
trestbps       0.037662 0.004574
     fbs       0.002512 0.000000

Top Random Forest Features:
Feature  Random Forest
     ca       0.198878
     cp       0.197879
thalach       0.127055
oldpeak       0.101776
  exang       0.084933

Top XGBoost Features:
Feature  XGBoost
     cp 0.279240
   thal 0.212163
     ca 0.190668
oldpeak 0.077766
  exang 0.058787
